# 03 · Differentiation in Astrophysics (Formula‑First, with units)Differentiation links changing quantities (e.g., velocity from position, gradients in space). We begin each example by writing the **full equation**, defining **symbols and units**, then computing the derivative in Python.

## When to use- Convert a model to a rate (e.g., $r(t)\to v$)- Interpret how fast a field changes ($dT/dr$)- Relate log-slopes, e.g., spectral index from $F_\nu\propto \nu^\alpha$

---## Example A — Orbital kinematics from a radial law**Model:** $r(t)=r_0 + v_0 t + \tfrac{1}{2} a t^2$**Symbols & units**| Symbol | Meaning | Units ||---|---|---|| $r(t)$ | radial position | m || $v(t)=dr/dt$ | radial velocity | m·s$^{-1}$ || $a$ | constant acceleration | m·s$^{-2}$ |**Derivative:** $v(t)=dr/dt=v_0 + a t$.

In [ ]:
import sympy as sp, numpy as npfrom sympy import lambdifyfrom astropy import units as ut = sp.symbols('t', nonnegative=True)r0, v0, a = 0.0, 20.0, 1.5  # SI numbers (attach units later)r = r0 + v0*t + sp.Rational(1,2)*a*t**2v = sp.diff(r, t)v_fn = lambdify(t, v, "numpy")ts = np.linspace(0, 10, 6) * u.sv_vals = v_fn(ts.value) * (u.m/u.s)v, v_vals

---## Example B — Temperature gradient in a star**Model:** $T(r)=T_0\left(1+\frac{r}{R}\right)^{-2}$**Symbols & units**| Symbol | Meaning | Units ||---|---|---|| $T(r)$ | temperature | K || $r$ | radius | m || $R$ | scale length | m |**Derivative:** $\dfrac{dT}{dr} = -\dfrac{2T_0}{R}\left(1+\dfrac{r}{R}\right)^{-3}$.

In [ ]:
import sympy as spfrom astropy import units as ur, T0, R = sp.symbols('r T0 R', positive=True)T = T0*(1 + r/R)**(-2)dTdr = sp.diff(T, r)T0_val = 1e7 * u.K; R_val = 5e7 * u.mgrad_at_R = dTdr.subs({T0:T0_val.value, R:R_val.value, r:R_val.value}) * (u.K/u.m)dTdr, grad_at_R

---## Example C — Spectral index as a log-derivative**Model:** $F_\nu(\nu)\propto \nu^\alpha$**Definition:** $\alpha = \dfrac{d\ln F_\nu}{d\ln \nu}$**Symbols & units**| Symbol | Meaning | Units ||---|---|---|| $F_\nu$ | flux density per frequency | W·m$^{-2}$·Hz$^{-1}$ || $\nu$ | frequency | Hz || $\alpha$ | spectral index | (dimensionless) |We verify numerically that $\alpha$ equals the slope of $\ln F_\nu$ vs $\ln \nu$.

In [ ]:
import numpy as npimport sympy as spfrom astropy import units as ualpha = 0.6nu = np.logspace(8, 12, 101) * u.HzF0 = 1e-26 * (u.W/(u.m**2*u.Hz))F = F0 * (nu/nu[0])**alpha# finite-difference log-slopelognu = np.log(nu.to_value(u.Hz))logF = np.log(F.to_value(u.W/(u.m**2*u.Hz)))slope = np.gradient(logF, lognu)alpha_est = slope.mean()alpha, alpha_est

## Exercises1) For the orbit model in Example A, compute a table of (t, v) at t=0…20 s in 5 s steps with units.2) For Example B, evaluate dT/dr at r=0, r=R, r=2R and comment on the trend.3) Generate F_ν with α=−0.7 and verify the average log-slope numerically.

## Solutions

In [ ]:
import numpy as np, sympy as spfrom sympy import lambdifyfrom astropy import units as u# 1t = sp.symbols('t', nonnegative=True)v = 20 + 1.5*tts = np.arange(0, 21, 5) * u.sv_vals = (20*u.m/u.s + 1.5*(u.m/u.s**2)*ts).to(u.m/u.s)# 2r = sp.symbols('r', positive=True); T0=1e7; R=5e7dTdr = sp.diff(T0*(1 + r/R)**(-2), r)pts = [0, R, 2*R]vals = [ (dTdr.subs({r:pt}) * (u.K/u.m)) for pt in pts ]# 3alpha = -0.7nu = np.logspace(8, 12, 301) * u.HzF0 = 1e-26 * (u.W/(u.m**2*u.Hz))F = F0 * (nu/nu[0])**alphaalpha_est = np.gradient(np.log(F.value), np.log(nu.value)).mean()ts, v_vals, vals, alpha_est